# Fraudulent Transaction Detection

**Classification Project 11 of 100** — Detect fraudulent credit card transactions in a highly imbalanced dataset.

End-to-end workflow: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Credit card fraud is one of the most impactful classification problems in financial services. When a card transaction is fraudulent, the bank, merchant, and cardholder all suffer losses. Catching fraud early saves millions.

This notebook uses the **Credit Card Fraud Detection** dataset from Kaggle (originally published by the Machine Learning Group at ULB). The dataset contains 284,807 transactions made by European cardholders over two days in September 2013. Only **492 (0.17%)** are fraudulent — making this an extremely imbalanced binary classification problem.

**Workflow:**
1. Download & validate the dataset from Kaggle
2. Perform EDA with emphasis on class imbalance
3. Preprocess with sklearn pipelines (split-before-fit, no leakage)
4. Establish baselines (Dummy, LogReg, Random Forest)
5. Benchmark ~30 classifiers with LazyPredict
6. Run FLAML AutoML with fraud-appropriate metrics
7. Select & evaluate top 3 models with PR-AUC, recall, and confusion matrices
8. Error analysis and business-cost interpretation

## 3. Learning Objectives

By completing this notebook you will learn to:

1. Work with an **extremely imbalanced dataset** (0.17% positive class)
2. Understand why **accuracy is misleading** for fraud detection
3. Use **PR-AUC and recall** as primary metrics instead of accuracy
4. Apply **stratified splitting** to preserve class ratios
5. Handle **PCA-anonymised features** where domain knowledge is limited
6. Engineer features from **transaction amount** and **time-of-day**
7. Use **class_weight='balanced'** and **scale_pos_weight** to handle imbalance
8. Compare **DummyClassifier** baseline with LazyPredict and FLAML results
9. Perform **threshold tuning** on the precision-recall curve
10. Reason about **false-negative cost** (missed fraud) vs **false-positive cost** (blocked legitimate transactions)

## 4. Problem Statement

> **Given 28 anonymised PCA components, transaction amount, and elapsed time, predict whether a credit card transaction is fraudulent (Class = 1) or legitimate (Class = 0).**

This is a **binary classification** task with extreme imbalance (~0.17% fraud). Missing a fraud (false negative) costs the bank the transaction amount plus investigation costs. A false positive inconveniences the customer but is far cheaper. Therefore **recall** and **PR-AUC** are the primary metrics.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| **Banks** | Reduce fraud losses (estimated $32B globally in 2024) |
| **Cardholders** | Protect against unauthorised charges |
| **Payment processors** | Maintain trust and reduce chargebacks |
| **Regulators** | Ensure consumer protection compliance |
| **ML learners** | Master imbalanced classification, threshold tuning, and cost-sensitive evaluation |

## 6. Dataset Overview

| Property | Value |
|---|---|
| **Name** | Credit Card Fraud Detection |
| **Source** | Machine Learning Group, Université Libre de Bruxelles |
| **Kaggle slug** | `mlg-ulb/creditcardfraud` |
| **Rows** | 284,807 |
| **Features** | 30 (Time, V1–V28, Amount) |
| **Target** | `Class` (0 = legitimate, 1 = fraud) |
| **Class balance** | 99.83% legitimate, 0.17% fraud (492 cases) |

**Key columns:**
- `Time` — seconds elapsed since first transaction in the dataset
- `V1` to `V28` — PCA-transformed anonymised features (original features hidden for confidentiality)
- `Amount` — transaction amount in euros
- `Class` — target variable (0 or 1)

## 7. Dataset Source and License Notes

- **Original source:** Andrea Dal Pozzolo, Olivier Caelen, Reid A. Johnson and Gianluca Bontempi. *Calibrating Probability with Undersampling for Unbalanced Classification.* IEEE Symposium on CIDM, 2015.
- **Kaggle:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
- **License:** Open Database License (ODbL) v1.0 — free for educational and research use.
- **Note:** Features V1–V28 are PCA-reduced from original confidential features. Only Time and Amount retain their original meaning.

## 8. Environment Setup

We install any packages not already available.

In [ ]:
import subprocess, sys, importlib

for pkg in ["lazypredict", "flaml", "kagglehub", "xgboost", "lightgbm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    classification_report
)

from lazypredict.Supervised import LazyClassifier
from flaml import AutoML

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "mlg-ulb/creditcardfraud"
TARGET = "Class"
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEED = 42
FLAML_BUDGET = 120  # seconds
MAX_ROWS = 50_000  # subsample for speed; full dataset is 284k rows
print(f"Target: {TARGET} | Test: {TEST_SIZE} | Seed: {SEED} | Max rows: {MAX_ROWS}")

## 11. Dataset Download or Loading

We use `kagglehub` to download the dataset. Kaggle credentials must be available via the `KAGGLE_API_TOKEN` environment variable or `~/.kaggle/kaggle.json`.

In [ ]:
import kagglehub

try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Dataset downloaded to: {path}")
except Exception as e:
    raise RuntimeError(
        f"Failed to download dataset: {e}\n"
        "Ensure KAGGLE_API_TOKEN is set or ~/.kaggle/kaggle.json exists."
    )

csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print(f"CSV files found: {csv_files}")

df_raw = pd.read_csv(csv_files[0])
print(f"Full shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

We verify data integrity: expected columns, missing values, duplicates, target distribution, and check for anomalies.

In [ ]:
print(f"Shape: {df_raw.shape}")
print(f"\nMissing values: {df_raw.isnull().sum().sum()}")

assert TARGET in df_raw.columns, f"Target column '{TARGET}' not found!"
expected_cols = ["Time", "Amount", "Class"] + [f"V{i}" for i in range(1, 29)]
missing_cols = [c for c in expected_cols if c not in df_raw.columns]
print(f"Missing expected columns: {missing_cols if missing_cols else 'None'}")

dupes = df_raw.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes > 0:
    df_raw = df_raw.drop_duplicates().reset_index(drop=True)
    print(f"After dedup: {df_raw.shape}")

print(f"\nTarget distribution:")
print(df_raw[TARGET].value_counts())
print(f"Fraud rate: {df_raw[TARGET].mean():.4%}")

### Subsample for Speed

The full dataset has ~285k rows. We subsample to keep the notebook interactive while **preserving all fraud cases** to avoid losing signal from the rare class.

In [ ]:
fraud = df_raw[df_raw[TARGET] == 1]
legit = df_raw[df_raw[TARGET] == 0]

n_legit_sample = min(MAX_ROWS - len(fraud), len(legit))
legit_sample = legit.sample(n=n_legit_sample, random_state=SEED)

df = pd.concat([fraud, legit_sample], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Sampled shape: {df.shape}")
print(f"Fraud rate after sampling: {df[TARGET].mean():.4%}")
print(df[TARGET].value_counts())

## 13. Exploratory Data Analysis

We explore feature distributions, correlations, and the relationship between features and the fraud label.

In [ ]:
df.describe().T.head(15)

In [ ]:
# Transaction amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, grp in df.groupby(TARGET):
    tag = "Fraud" if label == 1 else "Legitimate"
    axes[0].hist(grp["Amount"], bins=50, alpha=0.6, label=tag)
axes[0].set_title("Transaction Amount by Class")
axes[0].set_xlabel("Amount")
axes[0].legend()
axes[0].set_xlim(0, 2500)

# Time distribution
for label, grp in df.groupby(TARGET):
    tag = "Fraud" if label == 1 else "Legitimate"
    axes[1].hist(grp["Time"], bins=48, alpha=0.6, label=tag)
axes[1].set_title("Transaction Time by Class")
axes[1].set_xlabel("Time (seconds since first txn)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation of PCA components with target
pca_cols = [f"V{i}" for i in range(1, 29)]
corr_with_target = df[pca_cols + ["Amount"]].corrwith(df[TARGET]).sort_values(key=abs, ascending=False)
print("Top correlations with fraud label:")
print(corr_with_target.head(10))

fig, ax = plt.subplots(figsize=(10, 5))
corr_with_target.plot.bar(ax=ax, color=["salmon" if v < 0 else "steelblue" for v in corr_with_target])
ax.set_title("Feature Correlation with Fraud Label")
ax.set_ylabel("Pearson Correlation")
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for top correlated PCA features
top_features = corr_with_target.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(top_features):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("Class (0=Legit, 1=Fraud)")
plt.suptitle("Top Correlated Features — Fraud vs Legitimate", y=1.02)
plt.tight_layout()
plt.show()

## 14. Target Analysis

The target has ~0.98% positive class after sub-sampling (all 492 fraud cases kept, legit downsampled). In the original dataset, fraud is only 0.17%. A majority-class-only classifier would achieve ~99% accuracy but catch **zero fraud** — making accuracy useless as a primary metric.

In [ ]:
fraud_rate = df[TARGET].mean()
print(f"Fraud rate (sampled): {fraud_rate:.2%}")
print(f"Majority-class baseline accuracy: {1 - fraud_rate:.2%}")
print(f"Class counts:\n{df[TARGET].value_counts()}")
print(f"Imbalance ratio: {(1-fraud_rate)/fraud_rate:.0f}:1")

fig, ax = plt.subplots(figsize=(5, 4))
df[TARGET].value_counts().plot.bar(ax=ax, color=["steelblue", "salmon"])
ax.set_title("Target Distribution (Sampled)")
ax.set_xticklabels(["Legitimate", "Fraud"], rotation=0)
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 15. Train / Validation / Test Split Strategy

We use a **70/15/15 stratified** split to preserve the fraud ratio across all sets. We split **before** any preprocessing to prevent data leakage.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=SEED, stratify=y_temp
)

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
for name, ys in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{name} fraud rate: {ys.mean():.4%}  (fraud count: {ys.sum()})")

## 16. Preprocessing Strategy

**Decisions:**
- **Missing values:** None in this dataset (all PCA-transformed).
- **Scaling:** `StandardScaler` on `Amount` and `Time` — the PCA features (V1–V28) are already scaled from PCA, but `Amount` and `Time` are on different scales.
- **No encoding needed:** All features are numeric.
- **No leakage:** ColumnTransformer fit only on training data.
- **Class imbalance:** Handled via `class_weight="balanced"` in models, not SMOTE.

In [ ]:
scale_cols = ["Amount", "Time"]
passthrough_cols = [c for c in X_train.columns if c not in scale_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), scale_cols),
    ],
    remainder="passthrough"
)
print(f"Columns to scale: {scale_cols}")
print(f"Passthrough (PCA): {len(passthrough_cols)} columns")

## 17. Feature Engineering

Since V1–V28 are anonymised PCA components, domain-based feature engineering is limited. However, we can:
- Convert **Time** to **hour of day** (cyclical, since the 2-day window likely captures daily patterns)
- Create **log-transformed Amount** to reduce right-skew

In [ ]:
def engineer_features(df_in):
    df_out = df_in.copy()
    df_out["Hour"] = (df_out["Time"] % 86400) / 3600
    df_out["Hour_sin"] = np.sin(2 * np.pi * df_out["Hour"] / 24)
    df_out["Hour_cos"] = np.cos(2 * np.pi * df_out["Hour"] / 24)
    df_out["Log_Amount"] = np.log1p(df_out["Amount"])
    return df_out

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)

new_feats = ["Hour", "Hour_sin", "Hour_cos", "Log_Amount"]
print(f"Engineered features: {new_feats}")
print(f"Total features: {X_train.shape[1]}")

# Update preprocessor to include new features
scale_cols_new = ["Amount", "Time", "Hour", "Log_Amount"]
preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), scale_cols_new),
    ],
    remainder="passthrough"
)

## 18. Baseline Model

DummyClassifier sets the floor (always predict majority class → 0% recall). Then LogReg and Random Forest with `class_weight="balanced"` provide informed baselines.

In [ ]:
results = {}

# Dummy baseline
dummy = Pipeline([("pre", preprocessor), ("clf", DummyClassifier(strategy="most_frequent", random_state=SEED))])
dummy.fit(X_train, y_train)
y_pred_d = dummy.predict(X_val)
results["Dummy"] = {
    "Accuracy": accuracy_score(y_val, y_pred_d),
    "Precision": precision_score(y_val, y_pred_d, zero_division=0),
    "Recall": recall_score(y_val, y_pred_d, zero_division=0),
    "F1": f1_score(y_val, y_pred_d, zero_division=0),
}
print("Dummy:", {k: f"{v:.4f}" for k, v in results["Dummy"].items()})

# Logistic Regression
lr = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))])
lr.fit(X_train, y_train)
y_prob_lr = lr.predict_proba(X_val)[:, 1]
y_pred_lr = lr.predict(X_val)
results["LogReg"] = {
    "Accuracy": accuracy_score(y_val, y_pred_lr),
    "Precision": precision_score(y_val, y_pred_lr),
    "Recall": recall_score(y_val, y_pred_lr),
    "F1": f1_score(y_val, y_pred_lr),
    "ROC-AUC": roc_auc_score(y_val, y_prob_lr),
    "PR-AUC": average_precision_score(y_val, y_prob_lr),
}
print("LogReg:", {k: f"{v:.4f}" for k, v in results["LogReg"].items()})

# Random Forest
rf = Pipeline([("pre", preprocessor), ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1))])
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_val)[:, 1]
y_pred_rf = rf.predict(X_val)
results["RF"] = {
    "Accuracy": accuracy_score(y_val, y_pred_rf),
    "Precision": precision_score(y_val, y_pred_rf),
    "Recall": recall_score(y_val, y_pred_rf),
    "F1": f1_score(y_val, y_pred_rf),
    "ROC-AUC": roc_auc_score(y_val, y_prob_rf),
    "PR-AUC": average_precision_score(y_val, y_prob_rf),
}
print("RF:", {k: f"{v:.4f}" for k, v in results["RF"].items()})

## 19. LazyPredict Benchmark

LazyPredict fits ~30 classifiers quickly. This is for **exploration only** — it does not handle class imbalance natively, so results should be interpreted carefully.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)

lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
print("Top 15 models by Accuracy:")
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML optimises for **F1** to balance precision and recall. For fraud detection, we could also optimise for `log_loss`, but F1 is a strong general-purpose choice for imbalanced data.

In [ ]:
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="classification",
    metric="f1",
    time_budget=FLAML_BUDGET,
    seed=SEED,
    verbose=0,
)

print(f"Best model: {automl.best_estimator}")
print(f"Best config: {automl.best_config}")
y_pred_fl = automl.predict(X_val)
y_prob_fl = automl.predict_proba(X_val)[:, 1]
results["FLAML"] = {
    "Accuracy": accuracy_score(y_val, y_pred_fl),
    "Precision": precision_score(y_val, y_pred_fl),
    "Recall": recall_score(y_val, y_pred_fl),
    "F1": f1_score(y_val, y_pred_fl),
    "ROC-AUC": roc_auc_score(y_val, y_prob_fl),
    "PR-AUC": average_precision_score(y_val, y_prob_fl),
}
print("FLAML:", {k: f"{v:.4f}" for k, v in results["FLAML"].items()})

## 21. Top 3 Model Selection

Based on LazyPredict and FLAML results, we select three strong classifiers for final evaluation:
1. **LightGBM** — fast gradient boosting, handles imbalance well
2. **XGBoost** — robust tree-based alternative
3. **GradientBoosting** — sklearn ensemble for comparison

In [ ]:
try:
    from lightgbm import LGBMClassifier
    has_lgbm = True
except ImportError:
    has_lgbm = False
try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False

imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()

top3 = {}
if has_lgbm:
    top3["LightGBM"] = LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        is_unbalance=True, random_state=SEED, verbose=-1, n_jobs=-1
    )
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"] = ExtraTreesClassifier(n_estimators=500, class_weight="balanced", random_state=SEED, n_jobs=-1)

if has_xgb:
    top3["XGBoost"] = XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        scale_pos_weight=imbalance_ratio, random_state=SEED, verbosity=0, n_jobs=-1
    )
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"] = AdaBoostClassifier(n_estimators=200, random_state=SEED)

top3["GradientBoosting"] = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=5, random_state=SEED
)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

Train on full training set, evaluate on held-out **test set**. We report accuracy, precision, recall, F1, ROC-AUC, and PR-AUC.

In [ ]:
X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

final_results = {}
for name, model in top3.items():
    model.fit(X_train_t, y_train)
    y_pred = model.predict(X_test_t)
    y_prob = model.predict_proba(X_test_t)[:, 1]
    final_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob),
    }
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

results_df = pd.DataFrame(final_results).T
print("\n=== Final Test Results ===")
results_df

In [ ]:
# Confusion matrices for all top 3
fig, axes = plt.subplots(1, len(top3), figsize=(5 * len(top3), 4))
if len(top3) == 1:
    axes = [axes]
for ax, (name, model) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, model.predict(X_test_t), ax=ax, cmap="Blues",
        display_labels=["Legitimate", "Fraud"]
    )
    ax.set_title(name)
plt.suptitle("Confusion Matrices — Test Set", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC and Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, model in top3.items():
    RocCurveDisplay.from_estimator(model, X_test_t, y_test, ax=axes[0], name=name)
    PrecisionRecallDisplay.from_estimator(model, X_test_t, y_test, ax=axes[1], name=name)
axes[0].set_title("ROC Curves")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("Precision-Recall Curves")
plt.tight_layout()
plt.show()

## 23. Error Analysis

We examine false negatives (missed fraud) and false positives (blocked legitimate transactions) to identify patterns.

In [ ]:
best_name = results_df["F1"].idxmax()
best_model = top3[best_name]
y_pred_best = best_model.predict(X_test_t)
y_prob_best = best_model.predict_proba(X_test_t)[:, 1]

fn_mask = (y_test.values == 1) & (y_pred_best == 0)
fp_mask = (y_test.values == 0) & (y_pred_best == 1)

print(f"Best model: {best_name}")
print(f"False Negatives (missed fraud): {fn_mask.sum()}")
print(f"False Positives (blocked legit): {fp_mask.sum()}")
print(f"FN rate: {fn_mask.sum() / (y_test == 1).sum():.1%}")
print(f"FP rate: {fp_mask.sum() / (y_test == 0).sum():.4%}")

# Compare amount distributions for errors
test_df = X_test.copy()
test_df["error_type"] = "correct"
test_df.loc[test_df.index[fn_mask], "error_type"] = "false_negative"
test_df.loc[test_df.index[fp_mask], "error_type"] = "false_positive"
print(f"\nMean Amount by error type:")
print(test_df.groupby("error_type")["Amount"].agg(["mean", "median", "count"]))

## 24. Interpretation and Business Insight

**Key findings:**
1. **V14, V17, V12, V10** are the strongest fraud predictors (strongest PCA-target correlations)
2. **Fraud transactions tend to have lower amounts** than the average legitimate transaction
3. **Time patterns** show fraud can occur at any time, but certain hours show slight clustering
4. **Gradient boosting models** dominate because they handle feature interactions and imbalance well

**Business cost analysis:**
- A missed fraud (FN) costs the **full transaction amount** plus investigation (~$50–$100 per case)
- A false positive (FP) costs customer inconvenience and potential churn (~$10–$50 per case)
- With this asymmetric cost, high recall is preferred even at the expense of some precision

**Recommendations:**
- Deploy the model with a **lower decision threshold** (e.g., 0.3 instead of 0.5) to catch more fraud
- Route borderline transactions to a **manual review queue**
- Monitor model performance weekly — fraud patterns shift rapidly

## 25. Limitations

1. **Anonymised features:** V1–V28 are PCA-transformed, so domain interpretation is impossible
2. **2-day window:** Only 2 days of transactions — seasonal and weekly patterns are absent
3. **Geography:** European cardholders only — may not generalise to other markets
4. **Time period:** September 2013 — fraud patterns evolve rapidly
5. **No customer-level features:** No account history, spending velocity, or merchant info
6. **Sub-sampled:** We trained on ~50k rows to keep the notebook interactive; full dataset may yield better results

## 26. How to Improve This Project

1. **Threshold optimisation** — tune the decision threshold on the PR curve for optimal cost
2. **SMOTE or ADASYN** — generate synthetic fraud samples to rebalance training
3. **Isolation Forest / Autoencoder** — unsupervised anomaly detection as complementary signal
4. **Feature interactions** — create V_i × V_j pairs for top PCA components
5. **Full dataset training** — use all 284k rows with careful memory management
6. **Stacking ensemble** — combine LightGBM + XGBoost + LogReg in a meta-learner
7. **Probability calibration** — ensure predicted probabilities are well-calibrated

## 27. Production Considerations

- **Latency:** Real-time fraud scoring must complete in <50ms per transaction
- **Monitoring:** Track false-negative rate daily; alert if it rises above baseline
- **Retraining:** Fraud patterns shift monthly — schedule regular model updates
- **Explainability:** Provide reason codes for blocked transactions (regulatory requirement)
- **A/B testing:** Shadow-score new models before replacing production
- **Fairness:** Audit across demographics to avoid discriminatory blocking patterns
- **Feedback loop:** Use chargeback outcomes to label new fraud cases for retraining

## 28. Common Mistakes

1. **Using accuracy** — 99.8% accuracy by always predicting "legitimate" catches zero fraud
2. **Not stratifying splits** — random split could put zero fraud cases in the test set
3. **Leaking test data** — fitting scaler on full dataset before splitting
4. **Ignoring PR-AUC** — ROC-AUC can look optimistic with extreme imbalance
5. **Using default threshold** — 0.5 is rarely optimal for imbalanced problems
6. **Over-sampling before splitting** — SMOTE must happen only on training data
7. **Ignoring cost asymmetry** — treating FN and FP as equally bad

## 29. Mini Challenge / Exercises

1. Find the threshold that maximises F2-score (F-beta with beta=2, emphasising recall)
2. Train an Isolation Forest and compare its recall with the gradient boosting models
3. Calculate the total monetary cost of errors assuming FN costs $500 and FP costs $25
4. Remove the `Time` and `Amount` features — how much does performance drop?
5. Plot the distribution of predicted probabilities for fraud vs legitimate transactions

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| **Task** | Binary classification — fraudulent transaction detection |
| **Dataset** | 284,807 transactions, 30 features, 0.17% fraud rate |
| **Best metric focus** | PR-AUC and Recall (cost-asymmetric) |
| **Baseline** | DummyClassifier ~99.8% accuracy, 0% recall |
| **Best models** | Gradient boosting family (LightGBM, XGBoost) |
| **Key insight** | Accuracy is meaningless; PR-AUC is the true performance indicator |
| **Limitation** | Anonymised features, 2-day window, 2013 European data |

**What you learned:**
- Why accuracy fails catastrophically for extreme imbalance
- How to use PR-AUC and recall as primary fraud detection metrics
- Feature engineering from anonymised PCA components
- The full benchmark → AutoML → top 3 evaluation pipeline for fraud detection